In [1]:
import math
import os
import scipy
from scipy.optimize import lsq_linear
import numpy as np
from scipy.linalg import null_space
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal, halfnorm
import random
from scipy.io import loadmat
import random
import pickle
import sys
from sklearn.linear_model import RidgeCV
sys.path.append(r"c:\Users\katie\OneDrive\Documents\GitHub\trial")
import PCA_Regress as pcar
from brokenaxes import brokenaxes
from matplotlib.gridspec import GridSpec

In [2]:
base_path =r"c:\Users\katie\OneDrive\Desktop\Thesis"
with open(base_path+'\J_neu.pkl', "rb") as input_file:
    J_pickle = pickle.load(input_file)
del input_file

file_path = os.path.join(base_path, 'N_neu.pkl')
with open(file_path, "rb") as input_file:
    N_pickle = pickle.load(input_file)
del input_file

base_path =r"c:\Users\katie\OneDrive\Desktop\Thesis"
with open(base_path+'\J_mus.pkl', "rb") as input_file:
    J_pickle_m = pickle.load(input_file)
del input_file

ile_path = os.path.join(base_path, 'N_mus.pkl')
with open(ile_path, "rb") as input_file:
    N_pickle_m = pickle.load(input_file)
del input_file

# base_path = "/Users/kb6113/Desktop/Thesis"
# with open(base_path+'/J_neu.pkl', "rb") as input_file:
#     J_pickle = pickle.load(input_file)
# del input_file

# with open(base_path+'/J_mus.pkl', "rb") as input_file:
#     J_pickle_m = pickle.load(input_file)
# del input_file

J_all_tensor = J_pickle['J_all']['interpPSTH']
J_M1_tensor = J_pickle['J_M1']['interpPSTH']
J_PMd_tensor = J_pickle['J_PMd']['interpPSTH']
J_idx = np.r_[0:18, 36:45]
J_ntm_tensor = J_all_tensor[J_idx, :, :]
J_mus_tensor = J_pickle_m['interpPSTH']


N_all_tensor = N_pickle['N_all']['interpPSTH']
N_M1_tensor = N_pickle['N_M1']['interpPSTH']
N_PMd_tensor = N_pickle['N_PMd']['interpPSTH']
N_mus_tensor = N_pickle_m['interpPSTH']

<>:2: SyntaxWarning: invalid escape sequence '\J'
<>:12: SyntaxWarning: invalid escape sequence '\J'
<>:2: SyntaxWarning: invalid escape sequence '\J'
<>:12: SyntaxWarning: invalid escape sequence '\J'
C:\Users\katie\AppData\Local\Temp\ipykernel_2080\3800015342.py:2: SyntaxWarning: invalid escape sequence '\J'
  with open(base_path+'\J_neu.pkl', "rb") as input_file:
C:\Users\katie\AppData\Local\Temp\ipykernel_2080\3800015342.py:12: SyntaxWarning: invalid escape sequence '\J'
  with open(base_path+'\J_mus.pkl', "rb") as input_file:


In [6]:
def red_rank (tensor_N, tensor_M, rank): 
    """
    Running reduced rank regression 
    """

    # splicing preparatory and movement bins and scaling frmo 0 to 1 and mean centering
    Neu_pm, Neu_move, M_move = pcar.time_shift(tensor_N, tensor_M)

    # finding W_ls 
    cov = Neu_move.T @ Neu_move
    W_LS = np.linalg.solve(cov, Neu_move.T @ M_move)
    
    
    # running PCA on Neu_move @ W
    pre_PCA = W_LS.T @ Neu_move.T @ Neu_move @ W_LS 
    U, S_, V_T = np.linalg.svd(pre_PCA)
    V_rank = V_T.T[:,:rank]
    
    W_RRR = W_LS @ V_rank @ V_rank.T

    return W_RRR, V_rank, W_LS



In [42]:
rank = 3
W_RRR, V_rank, W_LS = red_rank(N_all_tensor, N_mus_tensor, rank)
cond = N_all_tensor.shape[0]
regress_N, Neu_move, regress_M = pcar.time_shift(N_all_tensor, N_mus_tensor)
U, S, V = np.linalg.svd(W_RRR)
print(regress_M.shape[1])
W_potent = U[:, :12]
W_null = U[:, 12:24]



12


In [40]:
# lengths of conditions x time, regress M only has movement, whereas regress_N has prep and movement
time_ct = regress_M.shape [0]
time_ct_neu = regress_N.shape [0]

# how many time bins are included in the movement period for a single condition
time_bins = int(time_ct / cond)

# how many time bins are included in the preparatory and movement period per condition
time_bins_pm = int(time_ct_neu / cond)

# difference in bins = just prep bins ie where the movement period starts for each condition
diff_bin = int((time_bins_pm - time_bins))

# isolating the preparatory and movement bins
N_tilde_tens = pcar.shape_tensor(regress_N, cond, time_bins_pm)
N_tilde_tens_move = N_tilde_tens[:,:,diff_bin:]
N_tilde_tens_prep = N_tilde_tens[:,:,:diff_bin]

# reshape into matrices for tuning computation
neu_move = pcar.shape_matrix(N_tilde_tens_move)
neu_prep = pcar.shape_matrix(N_tilde_tens_prep)

In [43]:
# movement null and potent space for gamma
N_null_move = neu_move @ W_null
N_nm_tensor = pcar.shape_tensor(N_null_move, cond)
N_nm_tensor -= N_nm_tensor.mean(axis=0, keepdims=True)     # the other one to comment out
N_null_move = pcar.shape_matrix(N_nm_tensor)
null_move_frob = np.linalg.norm(N_null_move)**2
null_move_var = np.sum(np.var(N_null_move, axis=0))

N_pot_move = neu_move @ W_potent
N_pm_tensor = pcar.shape_tensor(N_pot_move, cond)
N_pm_tensor -= N_pm_tensor.mean(axis=0, keepdims=True)     # the one to comment out
N_pot_move = pcar.shape_matrix(N_pm_tensor)
pot_move_frob = np.linalg.norm(N_pot_move)**2
pot_move_var = np.sum(np.var(N_pot_move, axis=0))

# computing gamma which is a scaling factor
gamma = null_move_var / pot_move_var
gamma2 = null_move_frob / pot_move_frob

# Null and potent projections of movement neural data
N_null_prep = neu_prep @ W_null
N_np_tensor = pcar.shape_tensor(N_null_prep, cond)
N_np_tensor -= N_np_tensor.mean(axis=0, keepdims=True)
N_null_prep = pcar.shape_matrix(N_np_tensor)       # subtract columns for variance
null_prep_frob = np.linalg.norm(N_null_prep)**2
null_prep_var = np.sum(np.var(N_null_prep, axis=0))

N_pot_prep = neu_prep @ W_potent
N_pp_tensor = pcar.shape_tensor(N_pot_prep, cond)
N_pp_tensor -= N_pp_tensor.mean(axis=0, keepdims=True)
N_pot_prep = pcar.shape_matrix(N_pp_tensor)       # subtract columns for variance
pot_prep_frob = np.linalg.norm(N_pot_prep)**2
pot_prep_var = np.sum(np.var(N_pot_prep, axis=0))

# tuning ratio
var_tuning = (null_prep_var / pot_prep_var) * ( 1/ gamma )   # this is with using the sum of variance
frob_tuning = (null_prep_frob / pot_prep_frob) * (1 / gamma2 )   # this is with using the frobenius norm and not variance on the movement data

# fraction of prep in null space and potent space
null_fraction = null_prep_var / (null_prep_var + pot_prep_var)
pot_fraction  = pot_prep_var  / (null_prep_var + pot_prep_var)

print("potent move variance: ", pot_move_var)
print("null move variance: ", null_move_var)
print("null prep variance:", null_prep_var)
print("potent prep variance: ", pot_prep_var)
print("Tuning with variance: ", var_tuning)
print("Tuning with frob: ", frob_tuning)
print("Prep fraction: ",  null_fraction)

potent move variance:  0.21817079664053904
null move variance:  0.31592016080561447
null prep variance: 0.051773696863934716
potent prep variance:  0.09770979613092441
Tuning with variance:  0.3659235369463035
Tuning with frob:  0.3659235369463041
Prep fraction:  0.346350595819401
